In [ ]:
# Analysis of Content Category Performance in Top YouTube Channels (June 2024)
# Problem: Analyse differences in subscriber counts, total views, and upload volume across content categories among the top 500 most-subscribed YouTube channels as of June 2024.
# Target User: Small content creators, new channel operators, and junior MCN teams.
# User Value: Provide data-driven content positioning insights to help users choose suitable content fields for channel growth.

# 1. Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import re

# 2. Load local dataset
df = pd.read_csv("youtube.csv")
# Standardize column names to lowercase to avoid KeyError
df.columns = df.columns.str.lower()

# 3. Define function to convert K/M/B units to numeric values
def convert_to_numeric(value):
    # Handle empty/NaN values
    if pd.isna(value):
        return None
    value = str(value).strip()
    # Match number + unit (K/M/B)
    match = re.match(r"([\d.]+)([KMB]?)", value, re.IGNORECASE)
    if not match:
        return None
    num, unit = match.groups()
    num = float(num)
    # Convert based on unit
    if unit.upper() == 'K':
        return num * 1_000
    elif unit.upper() == 'M':
        return num * 1_000_000
    elif unit.upper() == 'B':
        return num * 1_000_000_000
    else:
        return num

# 4. Convert columns with units to numeric
df['subscribers'] = df['subscribers'].apply(convert_to_numeric)
df['views'] = df['views'].apply(convert_to_numeric)
df['total_number_of_videos'] = df['total_number_of_videos'].apply(convert_to_numeric)

# 5. Show basic dataset information
print("=== Dataset Overview ===")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

# 6. Data cleaning
df = df.drop_duplicates()
df = df.dropna(subset=["category", "subscribers", "views", "total_number_of_videos"])

df['category'] = df['category'].astype(str).str.strip().str.title()

invalid_categories = ["Toyoraljanahktv", "Arab Games Network", "Unknown"]
df = df[~df['category'].isin(invalid_categories)]

valid_categories = [
    "Music", "Entertainment", "Gaming", "Kids", "Movies",
    "News", "Education", "Sports", "Technology", "Food"
]
df = df[df['category'].isin(valid_categories)]

print("\n=== After Cleaning ===")
print("New shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("\nCleaned Category Counts:")
print(df['category'].value_counts())

# 7. Group by category and calculate key statistics
category_stats = df.groupby("category").agg(
    channel_count=("subscribers", "count"),
    avg_subscribers=("subscribers", "mean"),
    avg_total_views=("views", "mean"),
    avg_uploads=("total_number_of_videos", "mean")
).round(2)

category_stats['avg_views_per_video'] = (category_stats['avg_total_views'] / category_stats['avg_uploads']).round(2)
category_stats = category_stats.sort_values(by="avg_subscribers", ascending=False)

print("\n=== Final Statistics by Category ===")
print(category_stats)

In [ ]:
# 8. Visualization (Optimized Version)
# Chart 1: Average Subscribers (with median line)
plt.figure(figsize=(12,6))
category_stats['avg_subscribers'].sort_values().plot(kind='barh', color='#4472C4')
median_subs = df['subscribers'].median()
plt.axvline(median_subs, color='black', linestyle='--', label='Overall Median')
plt.title('Average Subscribers by Content Category')
plt.xlabel('Average Subscribers')
plt.ylabel('Content Category')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Average Total Views (with median line)
plt.figure(figsize=(12,6))
category_stats['avg_total_views'].sort_values().plot(kind='barh', color='#FF4D4D')
median_views = df['views'].median()
plt.axvline(median_views, color='black', linestyle='--', label='Overall Median')
plt.title('Average Total Views by Content Category')
plt.xlabel('Average Total Views')
plt.ylabel('Content Category')
plt.legend()
plt.tight_layout()
plt.show()

# Chart 3: Average Upload Volume (with median line)
plt.figure(figsize=(12,6))
category_stats['avg_uploads'].sort_values().plot(kind='barh', color='#5CB85C')
median_uploads = df['total_number_of_videos'].median()
plt.axvline(median_uploads, color='black', linestyle='--', label='Overall Median')
plt.title('Average Upload Volume by Content Category')
plt.xlabel('Average Number of Uploads')
plt.ylabel('Content Category')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Average Views Per Video
plt.figure(figsize=(12,6))
category_stats['avg_views_per_video'].sort_values().plot(kind='barh', color='#9966FF')
plt.title('Average Views Per Video by Content Category')
plt.xlabel('Average Views Per Video')
plt.ylabel('Content Category')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: Upload Count vs Total Views (Log Scale Optimized)
plt.figure(figsize=(12,6))
for cat in df['category'].unique():
    subset = df[df['category'] == cat]
    plt.scatter(subset['total_number_of_videos'], subset['views'], label=cat, alpha=0.6)
plt.title('Upload Count vs Total Views by Content Category')
plt.xlabel('Total Number of Videos')
plt.ylabel('Total Views')
plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 6: Subscriber Distribution Boxplot
plt.figure(figsize=(14,7))
df.boxplot(column='subscribers', by='category', vert=False, patch_artist=True)
plt.title('Subscriber Distribution by Content Category')
plt.suptitle('')
plt.xlabel('Subscribers')
plt.ylabel('Content Category')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 7: Top 10 Channels Subscriber Share (with sample count label)
top10_share = []
categories = []
for cat in df['category'].unique():
    cat_data = df[df['category'] == cat].sort_values('subscribers', ascending=False)
    total = cat_data['subscribers'].sum()
    top10_sum = cat_data.head(10)['subscribers'].sum()
    share = (top10_sum / total) * 100
    categories.append(cat)
    top10_share.append(share)

top10_df = pd.DataFrame({
    'category': categories,
    'top10_share': top10_share
}).sort_values('top10_share')

plt.figure(figsize=(12,6))
bars = plt.barh(top10_df['category'], top10_df['top10_share'], color='#FF9933')
plt.title('Top 10 Channels Subscriber Share by Content Category')
plt.xlabel('Share of Total Subscribers (%)')
plt.ylabel('Content Category')

# Show category sample count
for i, bar in enumerate(bars):
    cat_name = top10_df['category'].iloc[i]
    count = df[df['category'] == cat_name].shape[0]
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'n={count}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 9. Final Conclusions
print("\n")
print("="*60)
print("FINAL CONCLUSIONS")
print("="*60)
print("1. Music and Entertainment categories have the highest average subscribers, making them the most audience-friendly choices for new creators to grow quickly.")
print("2. Entertainment and Gaming achieve the highest total views through frequent uploading, proving that consistency drives view growth for these categories.")
print("3. Some categories deliver high average views per video with low upload frequency, making them ideal for creators with limited time but strong content quality.")
print("4. The relationship between upload volume and views differs across categories, so creators should design custom update strategies instead of using a one-size-fits-all approach.")
print("5. Music shows a highly scattered subscriber distribution with strong head effects, so new creators face higher uncertainty in this category.")
print("6. Categories with high top-10 subscriber share are highly monopolized; new creators should avoid over-competitive categories to improve their chance of success.")
print("7. New creators should prioritize categories that match their ability to upload consistently, rather than only choosing popular fields.")
print("="*60)